In [ ]:
%load_ext autoreload
%autoreload 2

import anndata as ad
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

adata = ad.read_h5ad("./data/larry/larry_processed.h5ad")
adata

In [ ]:
import joblib
from scripts.VectorFieldEmbedder import VectorFieldEmbedder
from scripts.plotting import *

# Load the object back
emb = joblib.load("./data/larry/larry_embedder.pkl")
emb.gene_names = adata.var_names

print("Loaded object:", type(emb))

In [ ]:
from scripts.VectorFieldGeometry import FixedPointAnalyzer

# ---------------------------------------------------
# Run fixed point analysis
# ---------------------------------------------------
fpa = FixedPointAnalyzer(emb)

fp_info = fpa.identify_fixed_points(
    grid_size=60,
    speed_smooth_sigma=2.3,
    radius_percent=0.1,
)

print(f"Found {len(fp_info)} fixed points")

for i, info in enumerate(fp_info):
    print(f"\nFixed point {i}")
    print("  position:", info["position"])
    print("  type:", info["type"])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from sklearn.neighbors import NearestNeighbors
from scripts.plotting import compute_velocity_on_grid
from matplotlib.lines import Line2D

# --------------------------------------------------------------------------
# FIXED POINTS (DEFINE ONCE, USE EVERYWHERE)
# --------------------------------------------------------------------------
fp1 = fp_info[0]["position"]   # Fixed point 1 (source)
fp2 = fp_info[1]["position"]   # Fixed point 2 (saddle)

# --------------------------------------------------------------------------
# Helpers
# --------------------------------------------------------------------------
def points_inside_mask(X_emb, seeds, k=8, radius_scale=1.2):
    nn = NearestNeighbors(n_neighbors=k).fit(X_emb)
    r = np.median(nn.kneighbors(X_emb, n_neighbors=k)[0][:, -1]) * radius_scale
    neigh_idx = nn.radius_neighbors(seeds, radius=r, return_distance=False)
    return np.array([len(ix) > 0 for ix in neigh_idx])


def add_manual_arrows(ax, coords, X_emb, V_pred):
    nn = NearestNeighbors(n_neighbors=1).fit(X_emb)
    _, idx = nn.kneighbors(coords)
    idx = idx.ravel()

    ax.quiver(
        X_emb[idx, 0], X_emb[idx, 1],
        V_pred[idx, 0], V_pred[idx, 1],
        angles="xy", scale_units="xy", scale=3,
        width=0.003,
        headwidth=4.5, headlength=4.0, headaxislength=2.3,
        minlength=0.2,
        color="k", alpha=0.9
    )


# --------------------------------------------------------------------------
# 1) Embedding & labels
# --------------------------------------------------------------------------
X_emb = emb.X_emb
labels = np.asarray(adata.obs["state_info"].values)

# --------------------------------------------------------------------------
# 2) Colors
# --------------------------------------------------------------------------
uniq = np.unique(labels)
other = [lab for lab in uniq if lab != "Undifferentiated"]
cmap = plt.get_cmap("tab10", len(other))
colmap = {lab: mcolors.to_hex(cmap(i)) for i, lab in enumerate(other)}
colmap["Undifferentiated"] = "#d3d3d3"
cell_colors = np.array([colmap[lab] for lab in labels])

# --------------------------------------------------------------------------
# 3) Velocity grid
# --------------------------------------------------------------------------
Xg, keep_mass, _ = compute_velocity_on_grid(
    X_emb, grid_size=25, min_mass=0.01
)
keep_inside = points_inside_mask(X_emb, Xg)
Xg = Xg[keep_inside]
Vg = emb.tps_vf.predict(Xg)

# --------------------------------------------------------------------------
# 4) Plot
# --------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(12, 12))

# Background: Undifferentiated
if "Undifferentiated" in uniq:
    m = labels == "Undifferentiated"
    ax.scatter(
        X_emb[m, 0], X_emb[m, 1],
        c="#d3d3d3", s=40, alpha=0.2, linewidths=0
    )

# Foreground cells
m = labels != "Undifferentiated"
ax.scatter(
    X_emb[m, 0], X_emb[m, 1],
    c=cell_colors[m], s=80, alpha=0.4, linewidths=0
)

# Velocity field
ax.quiver(
    Xg[:, 0], Xg[:, 1],
    Vg[:, 0], Vg[:, 1],
    angles="xy", scale_units="xy", scale=3,
    width=0.003,
    headwidth=4.5, headlength=4.0, headaxislength=2.3,
    minlength=0.2,
    color="k", alpha=0.9
)

# --------------------------------------------------------------------------
# 5) Fixed points (explicit, no indices)
# --------------------------------------------------------------------------
dx, dy = -0.02, -0.03

# Fixed point 1
ax.scatter(fp1[0], fp1[1], color="red", s=1000, zorder=4)
ax.text(
    fp1[0] + dx, fp1[1] + dy, "1",
    color="white", fontsize=36, weight="bold",
    ha="center", va="center", zorder=5
)

# Fixed point 2
ax.scatter(fp2[0], fp2[1], color="red", s=1000, zorder=4)
ax.text(
    fp2[0] + dx, fp2[1] + dy, "2",
    color="white", fontsize=36, weight="bold",
    ha="center", va="center", zorder=5
)

# --------------------------------------------------------------------------
# 6) Manual arrows (optional patches)
# --------------------------------------------------------------------------
patch_coords = [
    [12.0, 10.4],
    [14.0, 4.0],
    [13.0, 4.0],
]

V_cells = emb.tps_vf.predict(X_emb)
add_manual_arrows(ax, patch_coords, X_emb, V_cells)

# --------------------------------------------------------------------------
# 7) Formatting
# --------------------------------------------------------------------------
ax.set_aspect("equal")
ax.set_xticks([]); ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
from matplotlib.ticker import MultipleLocator

uniq = np.unique(labels)
other = [lab for lab in uniq if lab != "Undifferentiated"]
cmap = plt.get_cmap("tab10", len(other))
colmap = {lab: mcolors.to_hex(cmap(i)) for i, lab in enumerate(other)}
colmap["Undifferentiated"] = "#d3d3d3"
cell_colors = np.array([colmap[lab] for lab in labels])

# Example patch coordinates (replace with your list)
patch_coords = [
    [-6.0,6.0],
    [2.3,-7.6]
]
X_emb = emb.X_emb
fig, ax = plt.subplots(figsize=(10, 10))

# Scatter all cells
ax.scatter(
    X_emb[:, 0], X_emb[:, 1],
    c=cell_colors, s=30, alpha=0.4, linewidths=0
)

# Overlay patch coordinates
patch_coords = np.array(patch_coords)
ax.scatter(
    patch_coords[:, 0], patch_coords[:, 1],
    c="red", s=80, marker="x", label="Patch coords"
)

# Equal aspect
ax.set_aspect("equal")

# Dense grid
ax.grid(True, linestyle="--", alpha=0.6, color="grey")
ax.xaxis.set_major_locator(MultipleLocator(1))   # grid every 1 unit
ax.yaxis.set_major_locator(MultipleLocator(1))   # grid every 1 unit
ax.minorticks_on()                               # finer ticks
ax.xaxis.set_minor_locator(MultipleLocator(0.5))
ax.yaxis.set_minor_locator(MultipleLocator(0.5))
ax.grid(which="minor", linestyle=":", alpha=0.3)

# Legend
ax.legend(fontsize=8, loc="upper right")

plt.tight_layout()
plt.show()

In [ ]:
from scripts.LAP import *

coords = np.array([
    [-6.0,6.0],
    [2.3,-7.6],
    fp2
])

lap = LagrangianPathOptimizer(
    vf=emb.tps_vf.predict,
    jacobian=emb.tps_vf.compute_jacobians,
    D=1.0,
    lam=1e-4
)

root = fp1

results = []
for i, end in enumerate(coords, start=1):
    print(f"\n=== Path {i}: {root} → {end} ===")
    res = lap.fit_path(
        X=emb.X_emb,
        V=emb.V_emb,
        X_orig=emb.X,
        distance_mode="orig",
        start=root,
        end=end,
        subsample_n=4000,
        n_segments=100,
        lr=5e-3,
        iters=300,
        plot=True,
        color=cell_colors
    )
    results.append(res)

# Collect refined paths
refined_paths = [r["path_refined"] for r in results]

In [ ]:
from scripts.plotting import *
from scripts.FieldReconstructionEvaluator import *

emb.fit_gene_level_splines(dof_gene=50, dof_vf_gene=50)
evaluator = FieldReconstructionEvaluator(emb)
res = evaluator.evaluate_gene_fit()

In [ ]:
from scripts.GradientAnalyzer import GradientAnalyzer

analyzer = GradientAnalyzer(emb, verbose=True)

for path in refined_paths:
    res = analyzer.evaluate_path_reconstruction(
        path,
        epsilon=0.03,
        threshold_expression=None,
        threshold_velocity=None
    )

    if res is not None:
        neigh_idx = res["neighbor_indices"]
        X = analyzer.emb.X_emb

        plt.figure(figsize=(6, 6))
        plt.scatter(X[:, 0], X[:, 1], c="lightgray", s=8, alpha=0.3, label="All cells")
        plt.scatter(X[neigh_idx, 0], X[neigh_idx, 1],
                    c="dodgerblue", s=20, alpha=0.8, label="Neighborhood")
        plt.plot(path[:, 0], path[:, 1],
                 c="darkorange", lw=2, alpha=0.9, label="Path")

        plt.scatter(path[0, 0], path[0, 1], c="green", s=60, edgecolor="black", label="Start")
        plt.scatter(path[-1, 0], path[-1, 1], c="red", s=60, edgecolor="black", label="End")

        plt.xlabel("UMAP-1")
        plt.ylabel("UMAP-2")
        plt.legend(frameon=False)
        plt.tight_layout()
        plt.show()
    else:
        print("No neighbors found.")

In [ ]:
def plot_fixed_point_panel_with_paths(
    fp_idx, fp, X_2d, tps_vf, adata,
    refined_paths,
    epsilon=0.20,
    nx=20, ny=20,
    stream_density=0.7,
    arrowsize=1.5,
    line_scale=2.5,
    savepath=None,
    title=None,
    path_color="red",
    path_alpha=0.8,
    path_style="--"
):
    emb_range = np.ptp(X_2d, axis=0)
    x0, y0 = map(float, fp)
    eps_x = epsilon * emb_range[0]
    eps_y = epsilon * emb_range[1]
    xlim = (x0 - eps_x, x0 + eps_x)
    ylim = (y0 - eps_y, y0 + eps_y)

    # --- Subset cells ---
    mask = (
        (X_2d[:, 0] >= xlim[0]) & (X_2d[:, 0] <= xlim[1]) &
        (X_2d[:, 1] >= ylim[0]) & (X_2d[:, 1] <= ylim[1])
    )
    X_sub = X_2d[mask]
    if X_sub.size == 0:
        print(f"[⚠️] No points near FP{fp_idx} ({x0:.2f}, {y0:.2f}). Skipping.")
        return None

    # --- Cell coloring ---
    labels = np.asarray(adata.obs["state_info"].values)
    uniq = np.unique(labels)
    other = [lab for lab in uniq if lab != "Undifferentiated"]
    cmap = plt.get_cmap("tab10", len(other))
    colmap = {lab: mcolors.to_hex(cmap(i)) for i, lab in enumerate(other)}
    colmap["Undifferentiated"] = "#d3d3d3"

    labels_sub = labels[mask]
    colors_sub = np.array([colmap[lab] for lab in labels_sub])

    undiff_mask = labels_sub == "Undifferentiated"
    diff_mask = ~undiff_mask

    # --- Velocity grid ---
    Xg, keep, Vg, meshes = compute_velocity_on_grid(
        X_sub,
        tps_vf=tps_vf,
        grid_size=nx,
        grid_density=1.0,
        min_mass=0.01,
        return_mesh=True
    )

    xx, yy = meshes
    ny_, nx_ = yy.shape[0], xx.shape[1]
    gx = np.linspace(xx[0, 0], xx[0, -1], nx_, dtype=float)
    gy = np.linspace(yy[0, 0], yy[-1, 0], ny_, dtype=float)
    Vx = np.full((ny_, nx_), np.nan)
    Vy = np.full((ny_, nx_), np.nan)
    dx = (gx[-1] - gx[0]) / (nx_ - 1)
    dy = (gy[-1] - gy[0]) / (ny_ - 1)
    j_idx = np.clip(np.rint((Xg[:, 0] - gx[0]) / dx).astype(int), 0, nx_ - 1)
    i_idx = np.clip(np.rint((Xg[:, 1] - gy[0]) / dy).astype(int), 0, ny_ - 1)
    Vx[i_idx, j_idx] = Vg[:, 0]
    Vy[i_idx, j_idx] = Vg[:, 1]

    # --- Plot ---
    fig, ax = plt.subplots(figsize=(8, 7), facecolor="white")

    # Background (undiff)
    if np.any(undiff_mask):
        ax.scatter(
            X_sub[undiff_mask, 0], X_sub[undiff_mask, 1],
            color="#d3d3d3", s=70, alpha=0.5, edgecolors="none", zorder=1
        )

    # Foreground (diff)
    if np.any(diff_mask):
        ax.scatter(
            X_sub[diff_mask, 0], X_sub[diff_mask, 1],
            color=colors_sub[diff_mask], s=70, alpha=0.8, edgecolors="none", zorder=2
        )

    # --- Dashed LAP paths ---
    for path in refined_paths:
        ax.plot(path[:, 0], path[:, 1],
                path_style, lw=9, color=path_color, alpha=path_alpha, zorder=2.5)

    # --- Vector arrows ---
    skip = (slice(None, None, 2), slice(None, None, 2))
    ax.quiver(
        xx[skip], yy[skip],
        np.nan_to_num(Vx[skip]), np.nan_to_num(Vy[skip]),
        angles="xy", scale_units="xy", scale=2.5,
        width=0.006, headwidth=5.0, headlength=4.5,
        color="k", alpha=0.9, zorder=3
    )

    # --- Fixed point 1 marker (existing) ---
    dx = -0.03
    dy = -0.08
    ax.scatter(x0, y0, color="red", s=2000, zorder=4)
    ax.text(
        x0 + dx, y0 + dy, str(fp_idx),
        color="white", fontsize=50, weight="bold",
        ha="center", va="center", zorder=5
    )

    # =========================================================
    # 🚨 DUCT-TAPED ADDITION: annotate Fixed Point 2
    # =========================================================
    x2, y2 = map(float, fp2)

    ax.scatter(
        x2, y2, color="red", s=2000, zorder=4
    )
    ax.text(
        x2 + dx, y2 + dy,
        "2",
        color="white",
        fontsize=50,
        weight="bold",
        ha="center",
        va="center",
        zorder=5
    )
    # =========================================================

    # --- Aesthetics ---
    ax.set_xlim(xlim); ax.set_ylim(ylim)
    ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
    ax.set_frame_on(False)
    ax.set_title(title or f"Fixed Point {fp_idx}", fontsize=36, pad=12)

    plt.tight_layout()
    if savepath:
        plt.savefig(savepath, dpi=300, bbox_inches="tight")
        print(f"[Saved] {savepath}")
    plt.show()

    return fig, ax



plot_fixed_point_panel_with_paths(
    fp_idx=1,
    fp=fp1,
    X_2d=emb.X_emb,
    tps_vf=emb.tps_vf,
    adata=adata,
    refined_paths=refined_paths,
    epsilon=0.3,
    path_alpha=0.9,
    path_color="darkorange",
    path_style="-",
    title="Fixed Point 1: Source",
    savepath="./figures/larry/fp1_source.pdf"
)

In [ ]:
analyzer = GradientAnalyzer(emb, verbose=True)
results_all = []  # collect all per-path analysis outputs

for trajectory in refined_paths:
    # Step 1 — evaluate local field reconstruction near the trajectory
    res = analyzer.evaluate_path_reconstruction(
        trajectory,
        epsilon=0.04,
        threshold_expression=None,
        threshold_velocity=None,
    )

    if res is None:
        continue

    # Step 2 — compute relative gene gradients vs velocity field
    rel_angles, magnitudes = analyzer.compute_relative_vectors(
        res["neighbor_indices"],
        res["selected_genes"],
        weight="mag",
        method="vector",
    )

    # Step 3 — store everything for custom plotting later
    results_all.append({
        "trajectory": trajectory,
        "neighbor_indices": res["neighbor_indices"],
        "selected_genes": res["selected_genes"],
        "gene_names": res["gene_names"],
        "expr_corrs": res["expr_corrs"],
        "vel_corrs": res["vel_corrs"],
        "relative_angles": rel_angles,
        "magnitudes": magnitudes,
    })

In [ ]:
# Choose which trajectory to visualize
res = results_all[0]

neighbor_indices = res["neighbor_indices"]
relative_angles  = res["relative_angles"]
magnitudes       = res["magnitudes"]
selected_genes   = res["selected_genes"]
expr_corr        = res["expr_corrs"]
vel_corr         = res["vel_corrs"]

# --- Compute base quantities ---
v_mean = analyzer.emb.V_emb[neighbor_indices].mean(axis=0)
base_angle = np.arctan2(v_mean[1], v_mean[0])
scatter_angles = base_angle + relative_angles

# ✅ Correct per-gene color: average of (J_i @ v_i) over neighbor cells
J = analyzer.jacobians[neighbor_indices, :, :]      # (Nc, G, 2)
V = analyzer.emb.V_emb[neighbor_indices]            # (Nc, 2)
vals_per_cell = np.einsum('cgd,cd->cg', J, V)       # (Nc, G)
vals = vals_per_cell.mean(axis=0)                   # (G,)
vals = vals[selected_genes]
# V_raw = analyzer.emb.V_raw[neighbor_indices][:, selected_genes]
# vals = V_raw.mean(axis=0)

cap = np.percentile(np.abs(vals), 99)
vals_clip = np.clip(vals, -cap, cap)

fit_quality = np.clip(expr_corr + vel_corr, 0, np.percentile(expr_corr + vel_corr, 99))
sizes = 1500 * (fit_quality / fit_quality.max() + 0.1)

# --- Plot base (no annotations) ---
fig, ax = plt.subplots(subplot_kw={'projection': 'polar'}, figsize=(8, 8))
ax.set_theta_zero_location('E')
ax.set_theta_direction(1)
ax.set_xticklabels([])
ax.set_yticklabels([])
ax.set_rticks([])
# Thicker and darker polar grid + axis lines
ax.xaxis.grid(True, linestyle='--', color='dimgray', alpha=0.5, lw=2)
ax.yaxis.grid(True, linestyle='--', color='dimgray', alpha=0.5, lw=2)
for spine in ax.spines.values():
    spine.set_color('black')
    spine.set_linewidth(2.5)

# Velocity direction
ax.plot([0, base_angle],
        [0, min(np.max(magnitudes), np.linalg.norm(v_mean))],
        color="black", lw=4, alpha=0.7)

# ---------------------------------------------------------------------
# Print top genes (within ±30° forward/reverse, top N by magnitude)
# ---------------------------------------------------------------------
deg2rad = np.pi / 180
angles_deg = np.degrees(relative_angles)
angles_norm = (relative_angles + 2*np.pi) % (2*np.pi)

forward_mask = (angles_norm <= 30 * deg2rad) | (angles_norm >= 2*np.pi - 30 * deg2rad)
reverse_mask = (angles_norm >= (180 - 30) * deg2rad) & (angles_norm <= (180 + 30) * deg2rad)
combined_mask = forward_mask | reverse_mask
candidate_indices = np.where(combined_mask)[0]
top_idx = candidate_indices[np.argsort(magnitudes[candidate_indices])[-12:]]

print("Top candidate genes to annotate manually:\n")
for idx in top_idx[::-1]:  # highest first
    direction = "forward" if forward_mask[idx] else "reverse"
    gname = analyzer.emb.gene_names[selected_genes[idx]]
    ang = angles_deg[idx]
    # Center angles around velocity direction for readability (-180 to 180)
    if ang > 180:
        ang -= 360
    print(f"{gname:20s} | magnitude={magnitudes[idx]:.3f} | angle={ang:7.2f}° | direction={direction}")


# ---------------------------------------------------------------------
# OPTIONAL: text annotations (comment out later if needed)
# ---------------------------------------------------------------------
annotate_idx = top_idx  # or any index list you like

for idx in annotate_idx:
    theta = scatter_angles[idx]
    r = magnitudes[idx]

    gname = analyzer.emb.gene_names[selected_genes[idx]]

    ax.text(
        theta,
        r * 1.05,              # small radial offset
        gname,
        fontsize=9,
        ha="center",
        va="center",
        color="black",
        alpha=0.9,
        clip_on=True,
    )

# Scatter points
sc = ax.scatter(
    scatter_angles, magnitudes,
    s=sizes, c=vals_clip,
    cmap='coolwarm', vmin=-cap, vmax=cap,
    alpha=0.7, edgecolors="grey"
)

save_dir = "./figures/larry"
os.makedirs(save_dir, exist_ok=True)

save_path = os.path.join(save_dir, "polar_gene_trajectory_a.png")

plt.tight_layout(pad=0.5)
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Figure saved to: {save_path}")


In [ ]:
# Choose which trajectory to visualize
res = results_all[1]

neighbor_indices = res["neighbor_indices"]
relative_angles  = res["relative_angles"]
magnitudes       = res["magnitudes"]
selected_genes   = res["selected_genes"]
expr_corr        = res["expr_corrs"]
vel_corr         = res["vel_corrs"]

# --- Compute base quantities ---
v_mean = analyzer.emb.V_emb[neighbor_indices].mean(axis=0)
base_angle = np.arctan2(v_mean[1], v_mean[0])
scatter_angles = base_angle + relative_angles

# ✅ Correct per-gene color: average of (J_i @ v_i) over neighbor cells
J = analyzer.jacobians[neighbor_indices, :, :]      # (Nc, G, 2)
V = analyzer.emb.V_emb[neighbor_indices]            # (Nc, 2)
vals_per_cell = np.einsum('cgd,cd->cg', J, V)       # (Nc, G)
vals = vals_per_cell.mean(axis=0)                   # (G,)
vals = vals[selected_genes]

cap = np.percentile(np.abs(vals), 99)
vals_clip = np.clip(vals, -cap, cap)

fit_quality = np.clip(expr_corr + vel_corr, 0, np.percentile(expr_corr + vel_corr, 99))
sizes = 1500 * (fit_quality / fit_quality.max() + 0.1)

# --- Plot base (no annotations) ---
fig, ax = plt.subplots(subplot_kw={'projection': 'polar'}, figsize=(8, 8))
ax.set_theta_zero_location('E')
ax.set_theta_direction(1)
ax.set_xticklabels([])
ax.set_yticklabels([])
ax.set_rticks([])
# Thicker and darker polar grid + axis lines
ax.xaxis.grid(True, linestyle='--', color='dimgray', alpha=0.5, lw=2)
ax.yaxis.grid(True, linestyle='--', color='dimgray', alpha=0.5, lw=2)
for spine in ax.spines.values():
    spine.set_color('black')
    spine.set_linewidth(2.5)

# Velocity direction
ax.plot([0, base_angle],
        [0, min(np.max(magnitudes), np.linalg.norm(v_mean))],
        color="black", lw=4, alpha=0.7)

# Scatter points
sc = ax.scatter(
    scatter_angles, magnitudes,
    s=sizes, c=vals_clip,
    cmap='coolwarm', vmin=-cap, vmax=cap,
    alpha=0.7, edgecolors="grey"
)

# ---------------------------------------------------------------------
# Print top genes (within ±30° forward/reverse, top N by magnitude)
# ---------------------------------------------------------------------
deg2rad = np.pi / 180
angles_deg = np.degrees(relative_angles)
angles_norm = (relative_angles + 2*np.pi) % (2*np.pi)

forward_mask = (angles_norm <= 30 * deg2rad) | (angles_norm >= 2*np.pi - 30 * deg2rad)
reverse_mask = (angles_norm >= (180 - 30) * deg2rad) & (angles_norm <= (180 + 30) * deg2rad)
combined_mask = forward_mask | reverse_mask
candidate_indices = np.where(combined_mask)[0]
top_idx = candidate_indices[np.argsort(magnitudes[candidate_indices])[-12:]]

print("Top candidate genes to annotate manually:\n")
for idx in top_idx[::-1]:  # highest first
    direction = "forward" if forward_mask[idx] else "reverse"
    gname = analyzer.emb.gene_names[selected_genes[idx]]
    ang = angles_deg[idx]
    # Center angles around velocity direction for readability (-180 to 180)
    if ang > 180:
        ang -= 360
    print(f"{gname:20s} | magnitude={magnitudes[idx]:.3f} | angle={ang:7.2f}° | direction={direction}")


# ---------------------------------------------------------------------
# OPTIONAL: text annotations (comment out later if needed)
# ---------------------------------------------------------------------
annotate_idx = top_idx  # or any index list you like

for idx in annotate_idx:
    theta = scatter_angles[idx]
    r = magnitudes[idx]

    gname = analyzer.emb.gene_names[selected_genes[idx]]

    ax.text(
        theta,
        r * 1.05,              # small radial offset
        gname,
        fontsize=9,
        ha="center",
        va="center",
        color="black",
        alpha=0.9,
        clip_on=True,
    )

save_dir = "./figures/larry"
os.makedirs(save_dir, exist_ok=True)

save_path = os.path.join(save_dir, "polar_gene_trajectory_b.png")

plt.tight_layout(pad=0.5)
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# ------------------------------------------------------------
# 1️⃣  Color Legend (coolwarm colormap, symmetric about 0)
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(6, 1.0))
fig.subplots_adjust(bottom=0.5)

# match the same cap values from main plot
vmin, vmax = -cap, cap
norm = plt.Normalize(vmin=vmin, vmax=vmax)
cbar = plt.colorbar(
    plt.cm.ScalarMappable(norm=norm, cmap='coolwarm'),
    cax=ax, orientation='horizontal'
)

# remove ticks & labels for a clean gradient bar
cbar.ax.tick_params(size=0, labelsize=0)
cbar.outline.set_linewidth(2)
cbar.outline.set_edgecolor("None")

plt.tight_layout(pad=0.5)
plt.show()


# Same size scaling as your main plot
min_size = 12000 * 0.1
max_size = 12000

# Create representative sizes (small → large)
sizes_ref = np.linspace(min_size, max_size, 4)

# Horizontal layout of circles
fig, ax = plt.subplots(figsize=(8, 4))
ax.set_aspect("equal")

# x positions for the circles
x_positions = np.arange(len(sizes_ref))

for x, s in zip(x_positions, sizes_ref):
    ax.scatter(x, 0, s=s, color="lightcoral", alpha=1, edgecolor="black", lw=10)

# Remove all axes and ticks
ax.set_xlim(-0.5, len(sizes_ref) - 0.5)
ax.set_ylim(-0.5, 0.5)
ax.axis("off")

plt.tight_layout(pad=0.5)
plt.show()


In [ ]:
from scipy.ndimage import gaussian_filter1d
from scipy.cluster.hierarchy import linkage, leaves_list

def plot_path_expression_from_results(analyzer, res,
                                      top_n=10,
                                      sigma=15,
                                      cmap="mako",
                                      figsize=(10,2)):
    """
    Plot smoothed, clustered expression heatmap for the top N forward-aligned genes.
    Uses precomputed results_all entry to save time.
    """
    neighbor_indices = res["neighbor_indices"]
    relative_angles  = res["relative_angles"]
    magnitudes       = res["magnitudes"]
    selected_genes   = res["selected_genes"]

    # --- Select top N forward-aligned genes (±30°) ---
    deg2rad = np.pi / 180
    angles_norm = (relative_angles + 2*np.pi) % (2*np.pi)
    forward_mask = (angles_norm <= 30 * deg2rad) | (angles_norm >= 2*np.pi - 30 * deg2rad)
    forward_idx = np.where(forward_mask)[0]

    if len(forward_idx) == 0:
        print("[Skip] No forward-aligned genes found.")
        return

    # Rank forward-aligned genes by magnitude (descending)
    ranked_forward = forward_idx[np.argsort(magnitudes[forward_idx])[::-1]]
    
    # Take top_n + 1 so we have a backup
    candidate_idx = ranked_forward[:top_n + 1]
    
    gene_names = np.array(analyzer.emb.gene_names)[selected_genes[candidate_idx]]
    
    # Remove CCR9 if present, then truncate back to top_n
    mask = gene_names != "Ccr9"
    filtered_idx = candidate_idx[mask]
    
    # Ensure we still have top_n genes
    top_idx = filtered_idx[:top_n]
    genes = np.array(analyzer.emb.gene_names)[selected_genes[top_idx]]


    print(f"[Select] Using top {len(genes)} forward-aligned genes.")

    # --- Get expression matrix ---
    X = analyzer.emb.X_raw[neighbor_indices]
    X = X[:, selected_genes[top_idx]]

    # --- Project cells onto path & sort ---
    path_coords = np.array(res["trajectory"])
    X_emb = analyzer.emb.X_emb[neighbor_indices]
    dist_to_path = np.linalg.norm(X_emb[:, None, :] - path_coords[None, :, :], axis=2)
    path_position = np.argmin(dist_to_path, axis=1)
    sort_order = np.argsort(path_position)
    X_sorted = X[sort_order]

    # --- Normalize and smooth ---
    X_norm = (X_sorted - X_sorted.mean(axis=0)) / (X_sorted.std(axis=0) + 1e-8)
    X_smooth = gaussian_filter1d(X_norm, sigma=sigma, axis=0, mode="nearest")

    # --- Cluster genes by expression pattern ---
    if X_smooth.shape[1] > 1:
        gene_link = linkage(X_smooth.T, method="average", metric="correlation")
        gene_order = leaves_list(gene_link)
    else:
        gene_order = np.arange(X_smooth.shape[1])

    X_clustered = X_smooth[:, gene_order]
    genes_ordered = np.array(genes)[gene_order]

    # --- Plot heatmap with gene names ---
    plt.figure(figsize=figsize, facecolor="white", dpi=150)
    sns.heatmap(
        X_clustered.T,
        cmap=cmap,
        center=0,
        vmin=-1, vmax=1,
        cbar=False,
        xticklabels=False,
        yticklabels=genes_ordered,
        linewidths=0,
        linecolor=None
    )
    plt.yticks(fontsize=12)
    plt.xticks([])
    plt.box(False)
    plt.tight_layout(pad=0)
    plt.show()


# ------------------------------------------------------------------
# Plot all four trajectories using saved results_all
# ------------------------------------------------------------------
for i, res in enumerate(results_all[:2], 1):
    print(f"\n=== Path {i} ===")
    plot_path_expression_from_results(
        analyzer, res,
        top_n=5,
        sigma=2,
        cmap="viridis",
        figsize=(6,0.8)
    )


In [ ]:
res = results_all[1]   # monocyte

relative_angles = res["relative_angles"]   # radians, relative to mean velocity
magnitudes      = res["magnitudes"]
selected_genes  = res["selected_genes"]
angles = (relative_angles + 2*np.pi) % (2*np.pi)

deg = np.pi / 180
width = 45 * deg

sectors = {
    "aligned":        (0 - width, 0 + width),
    "anti_aligned":   (np.pi - width, np.pi + width),
    "orth_left":      (np.pi/2 - width, np.pi/2 + width),
    "orth_right":     (3*np.pi/2 - width, 3*np.pi/2 + width),
}

In [ ]:
def select_gene(angle_center_range):
    lo, hi = angle_center_range
    mask = (angles >= lo) & (angles <= hi)
    idxs = np.where(mask)[0]
    if len(idxs) == 0:
        return None
    return idxs[np.argmax(magnitudes[idxs])]


In [ ]:
picked = {}
for k, ang_range in sectors.items():
    idx = select_gene(ang_range)
    picked[k] = idx

print("Selected genes (monocyte path):\n")
for k, idx in picked.items():
    gname = analyzer.emb.gene_names[selected_genes[idx]]
    ang = np.degrees(relative_angles[idx])
    mag = magnitudes[idx]
    print(f"{k:15s} | {gname:15s} | angle={ang:7.1f}° | magnitude={mag:.3f}")

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe

# ---------------------------------------------------------------------
# Choose which trajectory to visualize (monocyte)
# ---------------------------------------------------------------------
res = results_all[1]

neighbor_indices = res["neighbor_indices"]
relative_angles  = res["relative_angles"]
magnitudes       = res["magnitudes"]
selected_genes   = res["selected_genes"]
expr_corr        = res["expr_corrs"]
vel_corr         = res["vel_corrs"]

# ---------------------------------------------------------------------
# Genes to highlight (from geometric analysis)
# ---------------------------------------------------------------------
highlight_genes = ["Mmp12", "Rrm2", "H2-Aa", "BC100530"]
gene_names = analyzer.emb.gene_names

highlight_idx = [
    np.where(gene_names[selected_genes] == g)[0][0]
    for g in highlight_genes
]

# ---------------------------------------------------------------------
# Compute base quantities
# ---------------------------------------------------------------------
v_mean = analyzer.emb.V_emb[neighbor_indices].mean(axis=0)
base_angle = np.arctan2(v_mean[1], v_mean[0])
scatter_angles = base_angle + relative_angles

# Correct per-gene color: ⟨J v⟩
J = analyzer.jacobians[neighbor_indices, :, :]   # (Nc, G, 2)
V = analyzer.emb.V_emb[neighbor_indices]         # (Nc, 2)
vals_per_cell = np.einsum('cgd,cd->cg', J, V)    # (Nc, G)
vals = vals_per_cell.mean(axis=0)
vals = vals[selected_genes]

cap = np.percentile(np.abs(vals), 99)
vals_clip = np.clip(vals, -cap, cap)

fit_quality = np.clip(
    expr_corr + vel_corr,
    0,
    np.percentile(expr_corr + vel_corr, 99)
)
sizes = 1500 * (fit_quality / fit_quality.max() + 0.1)

# Boost size for highlighted genes
sizes_highlight = sizes.copy()
sizes_highlight[highlight_idx] *= 1.8

# ---------------------------------------------------------------------
# Plot base polar figure
# ---------------------------------------------------------------------
fig, ax = plt.subplots(
    subplot_kw={'projection': 'polar'},
    figsize=(8, 8)
)

ax.set_theta_zero_location('E')
ax.set_theta_direction(1)
ax.set_xticklabels([])
ax.set_yticklabels([])
ax.set_rticks([])

# Grid + frame styling
ax.xaxis.grid(True, linestyle='--', color='dimgray', alpha=0.5, lw=2)
ax.yaxis.grid(True, linestyle='--', color='dimgray', alpha=0.5, lw=2)
for spine in ax.spines.values():
    spine.set_color('black')
    spine.set_linewidth(2.5)

# Velocity direction
ax.plot(
    [0, base_angle],
    [0, min(np.max(magnitudes), np.linalg.norm(v_mean))],
    color="black",
    lw=4,
    alpha=0.7,
)

# ---------------------------------------------------------------------
# Scatter points
# ---------------------------------------------------------------------
sc = ax.scatter(
    scatter_angles,
    magnitudes,
    s=sizes_highlight,
    c=vals_clip,
    cmap="coolwarm",
    vmin=-cap,
    vmax=cap,
    alpha=0.7,
    edgecolors="grey",
    linewidths=0.5,
)

# ---------------------------------------------------------------------
# Highlighted gene annotations (large text, bounded)
# ---------------------------------------------------------------------
r_max = np.max(magnitudes)

for idx in highlight_idx:
    theta = scatter_angles[idx]
    r = magnitudes[idx]
    gname = gene_names[selected_genes[idx]]

    r_text = min(r * 1.10, 0.96 * r_max)  # <-- clamp inside boundary

    txt = ax.text(
        theta,
        r_text,
        gname,
        fontsize=30,
        fontweight="bold",
        ha="center",
        va="center",
        color="black",
        clip_on=False,     # <-- important
        zorder=10,
    )

    # White outline for readability
    txt.set_path_effects([
        pe.withStroke(linewidth=4, foreground="white")
    ])

# ---------------------------------------------------------------------
# Save
# ---------------------------------------------------------------------
save_dir = "./figures/larry"
os.makedirs(save_dir, exist_ok=True)

save_path = os.path.join(save_dir, "polar_gene_trajectory_monocyte_highlight.png")

plt.tight_layout(pad=0.5)
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Figure saved to: {save_path}")

In [ ]:
import matplotlib.colors as colors
from matplotlib import cm

# --------------------------------------------------------------------------
# Genes selected by flow-alignment analysis (monocyte path)
# --------------------------------------------------------------------------
genes_to_plot = ["Mmp12", "Rrm2", "H2-Aa", "BC100530"]
titles = [
    "Mmp12 (aligned)",
    "Rrm2 (anti-aligned)",
    "H2-Aa (orthogonal)",
    "BC100530 (orthogonal)"
]

# --------------------------------------------------------------------------
# Get raw expression
# --------------------------------------------------------------------------
gene_names = adata.var_names.to_numpy()

X_raw = adata.X
if hasattr(X_raw, "toarray"):
    X_raw = X_raw.toarray()

# --------------------------------------------------------------------------
# Shared color scale (robust to outliers)
# --------------------------------------------------------------------------
all_expr = []
for gene in genes_to_plot:
    idx = np.where(gene_names == gene)[0][0]
    all_expr.append(X_raw[:, idx])

all_expr = np.concatenate(all_expr)

# Robust limits (important for clean visuals)
vmin = np.percentile(all_expr, 1)
vmax = np.percentile(all_expr, 99)

def truncate_colormap(cmap, minval=0.05, maxval=1.0, n=256):
    return colors.LinearSegmentedColormap.from_list(
        "trunc_" + cmap.name,
        cmap(np.linspace(minval, maxval, n))
    )

cmap_dark = truncate_colormap(cm.magma_r, minval=0.05)

# --------------------------------------------------------------------------
# Plot: expression on FlowMap embedding
# --------------------------------------------------------------------------
fig, axes = plt.subplots(
    1, len(genes_to_plot),
    figsize=(5 * len(genes_to_plot), 5),
    sharex=True, sharey=True
)

if len(genes_to_plot) == 1:
    axes = [axes]

for ax, gene, title in zip(axes, genes_to_plot, titles):
    idx = np.where(gene_names == gene)[0][0]

    sc = ax.scatter(
        emb.X_emb[:, 0],
        emb.X_emb[:, 1],
        c=X_raw[:, idx],
        cmap=cmap_dark,
        s=10,
        alpha=0.6,
        vmin=vmin,
        vmax=vmax,
        linewidths=0
    )

    ax.set_title(title, fontsize=22)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_frame_on(False)

# Optional shared colorbar (recommended for Supplementary)
# cbar = fig.colorbar(
#     sc, ax=axes, fraction=0.02, pad=0.02
# )
cbar.set_label("Expression", fontsize=14)

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Monocyte cells by label
# ------------------------------------------------------------
labels = np.asarray(adata.obs["state_info"].values)
mono_mask = labels == "Monocyte"
mono_idx = np.where(mono_mask)[0]

X_2d = emb.X_emb[mono_idx, :]

# ------------------------------------------------------------
# Genes to plot
# ------------------------------------------------------------
genes = ["Mmp12", "Rrm2", "H2-Aa", "BC100530"]
gene_names = emb.gene_names
gidxs = [np.where(gene_names == g)[0][0] for g in genes]

# ------------------------------------------------------------
# Smoothed expression + Jacobians (already computed ONCE)
# ------------------------------------------------------------
expr_smooth = emb.tps_gene.predict(X_2d)          # (N_mono, G)
jacobians   = emb.tps_gene.compute_jacobians(X_2d)  # (N_mono, G, 2)

# ------------------------------------------------------------
# Hacky outlier removal (do ONCE, shared across genes)
# ------------------------------------------------------------
# remove farthest + rightmost
r = np.linalg.norm(X_2d, axis=1)
idx_far   = np.argmax(r)
idx_right = np.argmax(X_2d[:, 0])

keep = np.ones(len(X_2d), dtype=bool)
keep[[idx_far, idx_right]] = False

# remove top + right again (extra hack, as requested)
i_top   = np.argmax(X_2d[keep][:, 1])
i_right = np.argmax(X_2d[keep][:, 0])
keep[np.where(keep)[0][[i_top, i_right]]] = False

# apply mask
X_2d        = X_2d[keep]
expr_smooth = expr_smooth[keep]
jacobians   = jacobians[keep]

# ------------------------------------------------------------
# Plot: 2×2 gradient streamplots
# ------------------------------------------------------------
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
axes = axes.flatten()

for ax, gene, gidx in zip(axes, genes, gidxs):
    V_grad = jacobians[:, gidx, :]          # (N, 2)
    color  = expr_smooth[:, gidx]           # (N,)

    plot_velocity_streamplot(
        X_2d=X_2d,
        V=V_grad,
        scatter_color=color,
        grid_size=60,
        grid_density=1.0,
        stream_density=0.8,
        scatter_size=10,
        scatter_alpha=0.2,
        arrowsize=1.2,
        streamline_thickness=3.5,
        cmap=cmap_dark,
        title=gene,
        show_axes=False,
        show_colorbar=False,
        ax=ax,
    )

plt.tight_layout()
plt.show()